In [18]:
import sys
print(sys.executable)

/home/piets/sirna_shrna_offtargets/.venv/bin/python


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("All imports successful")

All imports successful


In [ ]:
## Code block intended for siRNA guide verification using ORF dataset from TargetScan

#### Assess siRNA sequences, seeds, and 3'UTR matching (from TargetScan) 8mer from now, but include 7merA1 and -m8, and 6mer in time

In [ ]:
## Store siRNA sequence() for downstream processing

# insert sequence(s) into dictionary

sirna_sequences = {
    'siPARP1A_sirna': "CCAUUGAGCACUUCAUGAA",
    'siPARP1B_sirna': "GAUUUCAUCUGGUGUGAUA",
    'siPARP2A_sirna': "GAAGAAAUUCCUUGACAAA",
    'siPARP2B_sirna': "AAGUACUAUCUGAUUCAGCUA"
}


In [21]:
# Reverse compliment

from Bio.Seq import Seq

sirna_rc = {}

for name, sequence in sirna_sequences.items():
    rc_name = name.replace("_sirna", "_rc")
    reverse_complement = str(Seq(sequence).reverse_complement_rna())
    sirna_rc[rc_name] = reverse_complement

In [22]:
# validation

for rc_name, sequence in sirna_rc.items():
    print(f"{rc_name}: {sequence}")

siPARP1A_rc: UUCAUGAAGUGCUCAAUGG
siPARP1B_rc: UAUCACACCAGAUGAAAUC
siPARP2A_rc: UUUGUCAAGGAAUUUCUUC
siPARP2B_rc: UAGCUGAAUCAGAUAGUACUU


In [23]:
# We want to define an 8mer seed, which first requires isolation of position 2-8 of the guide strand (# this is in principle the 7mer seed)

sirna_seeds_7mer = {}

for rc_name, sequence in sirna_rc.items():
    seed_7mer_name = rc_name.replace('_rc', '_seed_7mer')
    seed_7mer = sequence[1:8]
    sirna_seeds_7mer[seed_7mer_name] = seed_7mer

for seed_7mer_name, sequence in sirna_seeds_7mer.items():
    print(f'{seed_7mer_name}: {sequence}')

siPARP1A_seed_7mer: UCAUGAA
siPARP1B_seed_7mer: AUCACAC
siPARP2A_seed_7mer: UUGUCAA
siPARP2B_seed_7mer: AGCUGAA


In [24]:
# Construct reverse complement for downstream alignment with gene off-targets

sirna_targets_7mer = {}

for seed_name, seed_sequence in sirna_seeds_7mer.items():
    target_name = seed_name.replace('_seed_7mer', '_target_7mer')
    target_sequence = str(Seq(seed_sequence).reverse_complement())

    sirna_targets_7mer[target_name] = target_sequence

for target_name, sequence in sirna_targets_7mer.items():
    print(f'{target_name}: {sequence}')

siPARP1A_target_7mer: TTCATGA
siPARP1B_target_7mer: GTGTGAT
siPARP2A_target_7mer: TTGACAA
siPARP2B_target_7mer: TTCAGCT


In [25]:
# An 8mer-A1 requires an adenosine at position 1 (A1) adjacent to the seed match, i.e. the seed rc should have an A immediately upstream of the seed

sirna_targets_8mer_A1 = {}

for target_name, sequence in sirna_targets_7mer.items():
    target_8mer_name = target_name.replace('_target_7mer', '_target_8mer_A1')
    target_8mer_sequence = 'A' + sequence

    sirna_targets_8mer_A1[target_8mer_name] = target_8mer_sequence

for target_name, sequence in sirna_targets_8mer_A1.items():
    print(f'{target_name}: {sequence}')


siPARP1A_target_8mer_A1: ATTCATGA
siPARP1B_target_8mer_A1: AGTGTGAT
siPARP2A_target_8mer_A1: ATTGACAA
siPARP2B_target_8mer_A1: ATTCAGCT


In [26]:
output_path = Path(
    "../data/processed/GSE114273 (siPARP1_siPARP2)/GSE114273_sirna_target_motifs_7mer.tsv"
)

sirna_targets_7mer_df = pd.DataFrame(
    list(sirna_targets_7mer.items()),
    columns=["Target_name", "Target_sequence"]
)

sirna_targets_7mer_df.to_csv(output_path, sep="\t", index=False)

In [4]:
# Load UTR file
utr_path = Path("../data/processed/human_3UTRs_clean.tsv.gz")

utr_data = pd.read_csv(
    utr_path,
    sep="\t",
    low_memory=False
)

# Basic inspection
print("Shape:", utr_data.shape)
print("\nColumns:")
print(utr_data.columns.tolist())

print(utr_data.duplicated().sum())
display(utr_data.head(5))

Shape: (28351, 7)

Columns:
['transcript_id', 'gene_id', 'gene_symbol', 'utr_sequence', 'utr_length', 'gene_id_clean', 'has_8mer']
0


,transcript_id,gene_id,gene_symbol,utr_sequence,utr_length,gene_id_clean,has_8mer
0,ENST00000000233.5,ENSG00000004059.6,ARF5,CCAGCCAGGGGCAGGCCCCTGATGCCCGGAAGCTCCTGCGTGCATC...,422,ENSG00000004059,False
1,ENST00000000412.3,ENSG00000003056.3,M6PR,ATTGCACTTTATATGTCCAGCCTCTTCCTCAGTCCCCCAAACCAAA...,1457,ENSG00000003056,False
2,ENST00000001008.4,ENSG00000004478.5,FKBP4,CCCCTCTCCACCAGCCCTACTCCTGCGGCTGCCTGCCCCCCAGTCT...,2163,ENSG00000004478,False
3,ENST00000001146.2,ENSG00000003137.4,CYP26B1,CCCAAGACCCACCCGCCTCAGCCCAGCCCAGGCAGCGGGGTGGTGG...,3001,ENSG00000003137,False
4,ENST00000002125.4,ENSG00000003509.11,NDUFAF7,TATTTCAGCTTGGACATTTTACCCTTCAGTCGGCCCAAGAAATCAA...,810,ENSG00000003509,False


In [ ]:
# Unique genes and transcripts
print("Unique genes:", utr_data["gene_id_clean"].nunique())
print("Unique transcripts:", utr_data["transcript_id"].nunique())

# Check sequence alphabet
print("Sequence characters:", set("".join(utr_data["utr_sequence"].head(100))))

# Inspect existing 8mer annotation (columns should be removed)
print(utr_data["has_8mer"].value_counts(dropna=False))

Unique genes: 19473
Unique transcripts: 28351
Sequence characters: {'G', 'A', 'C', 'T'}
has_8mer
False    28254
True        97
Name: count, dtype: int64


In [29]:
# Load and inspect gene_expression_data and 

gex_path = Path(
    "../data/processed/GSE114273 (siPARP1_siPARP2)/GSE114273_expression_data.tsv"
)

expression_data = pd.read_csv(
    gex_path,
    sep="\t",
    low_memory=False
)

# Basic inspection
print("Shape:", expression_data.shape)
print("\nColumns:")
print(expression_data.columns.tolist())

print(expression_data.duplicated().sum())
print(expression_data.head(5))

Shape: (19227, 12)

Columns:
['GeneID', 'EnsemblGeneID', 'siNC_rep1', 'siNC_rep2', 'siPARP1-A_rep1', 'siPARP1-A_rep2', 'siPARP1-B_rep1', 'siPARP1-B_rep2', 'siPARP2-A_rep1', 'siPARP2-A_rep2', 'siPARP2-B_rep1', 'siPARP2-B_rep2']
0
   GeneID    EnsemblGeneID  siNC_rep1  siNC_rep2  siPARP1-A_rep1  \
0   79501  ENSG00000186092    0.00000    0.00000         0.00000   
1  729759  ENSG00000284733    0.00000    0.00000         0.11700   
2   81399  ENSG00000284662    0.01957    0.03683         0.03803   
3  148398  ENSG00000187634   21.28000   23.77000        17.28000   
4   26155  ENSG00000188976   98.92000  111.50000       112.20000   

   siPARP1-A_rep2  siPARP1-B_rep1  siPARP1-B_rep2  siPARP2-A_rep1  \
0         0.00000         0.00000         0.00000         0.00000   
1         0.00000         0.00000         0.00000         0.00000   
2         0.02007         0.02057         0.02181         0.03558   
3        15.92000        25.34000        24.84000        20.35000   
4       109.70000

In [30]:
# Map UTR records to expressed genes

expressed_gene_ids = expression_data["EnsemblGeneID"].unique()

utr_data_filtered = utr_data[
    utr_data["gene_id_clean"].isin(expressed_gene_ids)
].copy()

In [31]:
print("Expression genes:", expression_data["EnsemblGeneID"].nunique())
print("Matched UTR genes:", utr_data_filtered["gene_id_clean"].nunique())
print("Matched UTR transcripts:", utr_data_filtered["transcript_id"].nunique())

Expression genes: 19227
Matched UTR genes: 18268
Matched UTR transcripts: 26931


In [32]:
unmatched_genes = expression_data[
    ~expression_data["EnsemblGeneID"].isin(
        utr_data["gene_id_clean"]
    )
]

print("Unmatched genes:", len(unmatched_genes))
display(unmatched_genes.tail(10))

Unmatched genes: 959


,GeneID,EnsemblGeneID,siNC_rep1,siNC_rep2,siPARP1-A_rep1,siPARP1-A_rep2,siPARP1-B_rep1,siPARP1-B_rep2,siPARP2-A_rep1,siPARP2-A_rep2,siPARP2-B_rep1,siPARP2-B_rep2
19156,246100,ENSG00000268651,0.14710,0.1384,0.42870,0.1508,0.6184,0.1639,0.2675,0.2629,0.0000,0.1389
19164,474382,ENSG00000274183,0.00000,0.0000,0.00000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
19165,8263,ENSG00000288722,13.23000,11.3200,14.01000,12.8100,13.2700,16.0700,9.3650,9.6790,18.5200,22.4100
19173,474381,ENSG00000277858,0.00000,0.0000,0.00000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1799
19174,474383,ENSG00000288709,7.90600,8.2030,9.74900,9.6650,8.8410,11.0700,6.0820,6.5210,13.9600,17.9000
19175,474384,ENSG00000277150,8.10900,8.5840,10.14000,9.7690,8.9480,11.2900,5.9900,6.6110,14.7000,17.9000
19176,83740,ENSG00000277745,0.00000,0.0000,0.00000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2067
19207,140032,ENSG00000280969,1.06700,0.1255,0.25920,0.4103,0.0000,0.4459,0.3638,0.5959,0.2784,0.6300
19225,389831,ENSG00000276256,4.22400,4.4950,3.39500,3.4290,4.7560,4.5940,3.3600,3.3920,3.1220,2.4220
19226,100288966,ENSG00000277630,0.02962,0.0000,0.02878,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [33]:
# save expression data mapped to UTR dataset

utr_data_filtered.to_csv(
    "../data/processed/GSE114273 (siPARP1_siPARP2)/GSE114273_expressed_genes_3UTR_mapped.tsv",
    sep="\t",
    index=False
)

In [43]:
# Scan UTR sequences for each 8mer-A1 motif

for motif_name, motif_sequence in sirna_targets_8mer_A1.items():
    utr_data_filtered[f"{motif_name}_count"] = (
        utr_data_filtered["utr_sequence"].str.count(motif_sequence)
    )

    utr_data_filtered[f"{motif_name}_present"] = (
        utr_data_filtered[f"{motif_name}_count"] > 0
    )

In [ ]:
# Inspect and validate (check below with visualization that the actual 8merA1 motif fits the UTR)

motif_columns = [
    col for col in utr_data_filtered.columns
    if "_count" in col or "_present" in col
]

display(
    utr_data_filtered[
        ["gene_id_clean", "transcript_id"] + motif_columns
    ].head(10)
)

,gene_id_clean,transcript_id,siPARP1A_target_8mer_A1_count,siPARP1A_target_8mer_A1_present,siPARP1B_target_8mer_A1_count,siPARP1B_target_8mer_A1_present,siPARP2A_target_8mer_A1_count,siPARP2A_target_8mer_A1_present,siPARP2B_target_8mer_A1_count,siPARP2B_target_8mer_A1_present
0,ENSG00000004059,ENST00000000233.5,0,False,0,False,0,False,0,False
1,ENSG00000003056,ENST00000000412.3,0,False,0,False,0,False,0,False
2,ENSG00000004478,ENST00000001008.4,0,False,0,False,0,False,0,False
3,ENSG00000003137,ENST00000001146.2,0,False,0,False,0,False,0,False
4,ENSG00000003509,ENST00000002125.4,0,False,0,False,0,False,0,False
5,ENSG00000001036,ENST00000002165.6,0,False,0,False,0,False,0,False
6,ENSG00000002587,ENST00000002596.5,0,False,0,False,0,False,0,False
7,ENSG00000001617,ENST00000002829.3,0,False,0,False,0,False,0,False
8,ENSG00000001630,ENST00000003100.8,1,True,1,True,0,False,0,False
9,ENSG00000048028,ENST00000003302.4,1,True,0,False,0,False,0,False


In [45]:
# Aggregrate transcript output to gene level

# Define aggregation operations
aggregation = {
    "transcript_id": "nunique"
}

for motif_name in sirna_targets_8mer_A1:
    aggregation[f"{motif_name}_present"] = "max"
    aggregation[f"{motif_name}_count"] = "sum"


# Aggregate by Ensembl gene ID
gene_utr_summary = (
    utr_data_filtered
    .groupby("gene_id_clean")
    .agg(aggregation)
    .reset_index()
)

# Rename transcript count column
gene_utr_summary = gene_utr_summary.rename(
    columns={"transcript_id": "n_utr_transcripts"}
)

In [46]:
# Merge with gene expression data

gene_expression_utr = expression_data.merge(
    gene_utr_summary,
    left_on="EnsemblGeneID",
    right_on="gene_id_clean",
    how="left"
)

gene_expression_utr = gene_expression_utr.drop(
    columns="gene_id_clean"
)

In [47]:
# Inspect

print("Shape:", gene_expression_utr.shape)
print("Unique genes:", gene_expression_utr["EnsemblGeneID"].nunique())

display(gene_expression_utr.head())

Shape: (19227, 21)
Unique genes: 19227


,GeneID,EnsemblGeneID,siNC_rep1,siNC_rep2,siPARP1-A_rep1,siPARP1-A_rep2,siPARP1-B_rep1,siPARP1-B_rep2,siPARP2-A_rep1,siPARP2-A_rep2,...,siPARP2-B_rep2,n_utr_transcripts,siPARP1A_target_8mer_A1_present,siPARP1A_target_8mer_A1_count,siPARP1B_target_8mer_A1_present,siPARP1B_target_8mer_A1_count,siPARP2A_target_8mer_A1_present,siPARP2A_target_8mer_A1_count,siPARP2B_target_8mer_A1_present,siPARP2B_target_8mer_A1_count
0,79501,ENSG00000186092,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,1.0,False,0.0,False,0.0,False,0.0,False,0.0
1,729759,ENSG00000284733,0.00000,0.00000,0.11700,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,81399,ENSG00000284662,0.01957,0.03683,0.03803,0.02007,0.02057,0.02181,0.03558,0.03497,...,0.03697,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,148398,ENSG00000187634,21.28000,23.77000,17.28000,15.92000,25.34000,24.84000,20.35000,18.93000,...,15.33000,1.0,False,0.0,False,0.0,False,0.0,False,0.0
4,26155,ENSG00000188976,98.92000,111.50000,112.20000,109.70000,111.50000,101.90000,147.10000,139.00000,...,130.80000,1.0,False,0.0,False,0.0,False,0.0,False,0.0


In [48]:
# Check seed-postive counts

for motif_name in sirna_targets_8mer_A1:
    column = f"{motif_name}_present"
    print(f"{motif_name}: {gene_expression_utr[column].sum()} positive genes")

siPARP1A_target_8mer_A1: 918 positive genes
siPARP1B_target_8mer_A1: 781 positive genes
siPARP2A_target_8mer_A1: 693 positive genes
siPARP2B_target_8mer_A1: 837 positive genes


In [ ]:
# Validate data is consistent (expect 959 NaN due to there being 959 expressed genes that were not mapped to any UTR)

for motif_name in sirna_targets_8mer_A1:
    column = f"{motif_name}_present"
    print(column)
    print(gene_expression_utr[column].value_counts(dropna=False))

siPARP1A_target_8mer_A1_present
siPARP1A_target_8mer_A1_present
False    17350
NaN        959
True       918
Name: count, dtype: int64
siPARP1B_target_8mer_A1_present
siPARP1B_target_8mer_A1_present
False    17487
NaN        959
True       781
Name: count, dtype: int64
siPARP2A_target_8mer_A1_present
siPARP2A_target_8mer_A1_present
False    17575
NaN        959
True       693
Name: count, dtype: int64
siPARP2B_target_8mer_A1_present
siPARP2B_target_8mer_A1_present
False    17431
NaN        959
True       837
Name: count, dtype: int64


In [50]:
output_path = Path(
    "../data/processed/GSE114273 (siPARP1_siPARP2)/GSE114273_expression_data_8mer_A1_annotated.tsv"
)

gene_expression_utr.to_csv(output_path, sep="\t", index=False)

####

### UTR Mapping and 8mer-A1 Seed Annotation

- Mapped the 3′UTR dataset to the 19,227 expressed protein-coding genes using Ensembl gene IDs.
- Identified 18,268 genes with at least one annotated 3′UTR transcript (26,931 transcripts in total).
- Scanned transcript sequences for the four siRNA-specific 8mer-A1 target motifs.
- Aggregated motif information to gene level. A gene was classified as seed-positive if at least one of its annotated 3′UTR transcripts contained the corresponding motif.
- Recorded both binary seed-match status and total motif occurrences across transcripts.

**Seed-positive genes:**
- siPARP1-A: 918
- siPARP1-B: 781
- siPARP2-A: 693
- siPARP2-B: 837

Genes without an annotated 3′UTR will be excluded from the downstream seed analysis.